# Dimension-based Engagement Predictor

This notebook creates a simple linear model to predict video engagement (`log_views`) based on the 20 PCA dimensions of the video embeddings per channel.
We aim to maximize dimensionality and understand the significance of each dimension for predicting performance.

## 1) Install dependencies and connect to Drive
We install required libraries, including `statsmodels` for linear modeling, and mount Google Drive if running in Colab.

In [1]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn statsmodels

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from pathlib import Path

# Set up matplotlib style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab, skipping drive mount')


Mounted at /content/drive


## 2) Load 20D Video Embeddings
We load the 20D video embeddings from the canonical path according to our coding guidelines. Then we filter out missing views and calculate `log_views`.

In [2]:
DATA_PATH = Path('/content/drive/MyDrive/Graphiko/exports/video_embeddings_reduced/latest/business_cluster_video_embeddings_reduced_20d.csv')

if not DATA_PATH.exists():
    print(f"Warning: {DATA_PATH} not found. Attempting to run with dummy data or fallback if available.")

if DATA_PATH.suffix.lower() == '.csv':
    try:
        df = pd.read_csv(DATA_PATH)
    except FileNotFoundError:
        df = pd.DataFrame() # Fallback
elif DATA_PATH.suffix.lower() == '.json':
    df = pd.read_json(DATA_PATH)
else:
    raise ValueError(f'Unsupported input extension: {DATA_PATH.suffix}')

dim_cols = []
if not df.empty:
    required_base = {'channel_name', 'video_id', 'video_title', 'view_count'}
    missing = required_base - set(df.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')

    dim_cols = [c for c in df.columns if c.startswith('embedding_reduced_')]
    if len(dim_cols) != 20:
        if 'embedding_20d' in df.columns:
            import ast
            if isinstance(df['embedding_20d'].iloc[0], str):
                df['embedding_20d'] = df['embedding_20d'].apply(ast.literal_eval)
            dim_cols = [f'dim_{i}' for i in range(20)]
            emb_df = pd.DataFrame(df['embedding_20d'].tolist(), columns=dim_cols, index=df.index)
            df = pd.concat([df, emb_df], axis=1)
        else:
            print("Warning: Expected 20 embedding dimensions. Ensure the data format is correct.")

    df['view_count'] = pd.to_numeric(df['view_count'], errors='coerce')
    df = df.dropna(subset=['channel_name', 'view_count']).copy()
    df['log_views'] = np.log1p(df['view_count'])

    print(f'Rows loaded: {len(df):,}')
    print(f'Channels: {df["channel_name"].nunique()}')


Rows loaded: 1,344
Channels: 27


## 3) Reusable OLS Function
We encapsulate the Ordinary Least Squares (OLS) regression logic into a reusable function `fit_ols_model`. This function takes a dataset (or a subset), fits the model using the 20 dimensions as predictors for `log_views`, and returns a list of dictionaries containing dimension significance and coefficients.
We also define a reusable plotting function `plot_dimension_impact` that creates a bar chart colored by the sign of the coefficient (blue for positive impact, red for negative).

In [ ]:
def fit_ols_model(group, dim_cols, channel_name='Global'):
    # We need more observations than predictors to fit OLS
    if len(group) <= len(dim_cols) + 1:
        return []
    
    X = group[dim_cols]
    y = group['log_views']
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    
    results = []
    # Extract significance (p-values) and coefficients
    for i, dim in enumerate(dim_cols):
        results.append({
            'channel_name': channel_name,
            'dimension': dim,
            'dimension_index': i,
            'coefficient': model.params[dim],
            'p_value': model.pvalues[dim],
            'is_significant': model.pvalues[dim] < 0.05,
            'r2_adj': model.rsquared_adj
        })
    return results

def plot_dimension_impact(summary_df, coef_col='coefficient', title='Dimension Impact on Engagement'):
    # Create a list of colors: blue for positive, red for negative
    colors = ['steelblue' if c > 0 else 'indianred' for c in summary_df[coef_col]]
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=summary_df, x=coef_col, y='dimension', palette=colors)
    plt.title(title)
    plt.xlabel('Coefficient Impact')
    plt.ylabel('Dimension')
    plt.tight_layout()
    plt.show()


## 4) Train Global Prediction Model
We use the `fit_ols_model` function on the entire dataset to compute a global prediction model. The results are sorted by the coefficient to prioritize how the dimension affects performance globally, and then we plot the dimension impact.

In [ ]:
global_results_df = pd.DataFrame()
if not df.empty and dim_cols:
    global_results = fit_ols_model(df, dim_cols, channel_name='Global')
    if global_results:
        global_results_df = pd.DataFrame(global_results)
        # Sort by coefficient descending to prioritize performance impact
        global_results_df = global_results_df.sort_values('coefficient', ascending=False)
        
        print("Global Prediction Model Results:")
        display(global_results_df[['dimension', 'coefficient', 'p_value', 'is_significant']])
        
        plot_dimension_impact(global_results_df, coef_col='coefficient', title='Global Dimension Impact on Engagement')
    else:
        print("Not enough data for global model.")


## 5) Train Linear Models per Channel
After computing the global prediction model, we reuse the algorithmic implementation to train one model per channel. We then aggregate the results to analyze how significant each dimension is across all channels, and how it affects performance on average (`avg_coef`).

In [ ]:
results = []
if not df.empty and dim_cols:
    for channel, group in df.groupby('channel_name'):
        channel_results = fit_ols_model(group, dim_cols, channel_name=channel)
        results.extend(channel_results)

    results_df = pd.DataFrame(results)
    if not results_df.empty:
        print(f"Modeled {results_df['channel_name'].nunique()} channels.")
        
        # Aggregate by dimension
        dim_summary = results_df.groupby('dimension').agg(
            significant_count=('is_significant', 'sum'),
            total_count=('channel_name', 'count'),
            avg_coef=('coefficient', 'mean')
        ).reset_index()

        dim_summary['significant_pct'] = (dim_summary['significant_count'] / dim_summary['total_count']) * 100
        
        # Sort by avg_coef to prioritize performance impact
        dim_summary = dim_summary.sort_values('avg_coef', ascending=False)
        
        print("\nAggregated Per-Channel Dimension Summary:")
        display(dim_summary.head(20))
    else:
        print("No valid per-channel models were trained.")
else:
    results_df = pd.DataFrame()
    print("No data available to train models.")


## 6) Per-Channel Charts and Visualization
We visualize the average impact of each dimension across channels (blue for positive, red for negative). We also show a heatmap of dimension significance (p-values) per channel.

In [ ]:
if not results_df.empty:
    # 1. Bar chart of average coefficient impact per dimension
    plot_dimension_impact(dim_summary, coef_col='avg_coef', title='Average Dimension Impact on Engagement Across Channels')

    # 2. Heatmap of p-values per channel and dimension (Top 20 channels by R2 adj)
    top_channels = results_df[['channel_name', 'r2_adj']].drop_duplicates().sort_values('r2_adj', ascending=False).head(20)['channel_name']
    heatmap_data = results_df[results_df['channel_name'].isin(top_channels)].pivot(index='channel_name', columns='dimension', values='p_value')

    # Sort columns by dimension number for better readability if they are named consistently
    try:
        sorted_cols = sorted(heatmap_data.columns, key=lambda x: int(x.split('_')[-1]))
        heatmap_data = heatmap_data[sorted_cols]
    except:
        pass

    plt.figure(figsize=(14, 10))
    sns.heatmap(heatmap_data, cmap='coolwarm_r', vmin=0, vmax=0.1, annot=False)
    plt.title('P-Value Heatmap (Red = Highly Significant, p < 0.05)')
    plt.tight_layout()
    plt.show()
